**Eye-Tracking & Neurocognitive Intelligence | Eye-Tracking Foundations & Signal Analysis | Presented by: Reza Saadatyar (2026)**

**02 | Gap-Aware Gaze Preprocessing and Velocity**

![Gap-aware preprocessing: measured stage-01 gaze input and method-specific valid-run, median and velocity diagrams](../../images/02_gap_aware_gaze_preprocessing.svg)

**Objective:** Produce an offline, gap-respecting gaze representation and adjacent-sample angular velocity for later fixation and saccade analysis.

**Input evidence:** Notebook 01 recorded **10 participants**, **40 task recordings** and **2,778,008 samples** from GazeBase Round 1, Session 1. This Notebook reads its saved stage-01 CSV rather than the original GazeBase archive.

**Method scope:** Process complete source records independently, retain original `n`, `x/y`, validity and source identifiers, and calculate smoothed coordinates and velocity only within eligible runs.

**Execution status:** Notebook 02 has **not** been executed on the full original ten-participant subset. Its figure includes a measured raw-source excerpt, but the illustrated segmentation, median and velocity operations are **method schematics**, not recorded Notebook 02 results.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_tracking_foundations")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# ------------------------ 5. Study Research Utilities ------------------------
from core.stage_outputs import ensure_stage_inputs
from gaze_streaming import preprocess_stage_file

# --------------------- 6. Output and Presentation Setup ----------------------
output_dir = root / "eye_tracking_foundations" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(10, 4),
)
print("Current outputs:", output_dir)


**02. Recover Notebook 01 Current Inputs**

Reuse `01_gaze_samples.csv` and `01_source_manifest.csv` from the completed source audit. The compact manifest identifies the expected 40 task recordings and ten participants; the sample CSV contains all 2,778,008 stage-01 rows.

If the files are present in `eye_tracking_foundations/output/`, reuse them unchanged. In a new Colab runtime, restore the matching `eye_tracking_foundations_Notebook_01_Outputs.zip` from `/content/`. The stage helper checks the named inputs, while the next cell reads the large CSV in bounded chunks and processes **one entire record at a time**. No source archive download or repeated Notebook 01 computation is performed.


In [ ]:
# ================== 02. Recover Notebook 01 Current Inputs ===================

# ----------------------- 1. Verify Upstream Outputs --------------------------
ensure_stage_inputs(
    output_dir, 1, ("01_gaze_samples.csv", "01_source_manifest.csv")
)
source_csv = output_dir / "01_gaze_samples.csv"
manifest = pd.read_csv(
    output_dir / "01_source_manifest.csv",
    dtype={"record_id": str, "participant_id": str}, low_memory=False,
)
if manifest["record_id"].duplicated().any():
    raise ValueError("Duplicate record identities in Notebook 01 manifest.")
expected_records = set(manifest["record_id"].astype(str))
print("Upstream participants:", manifest["participant_id"].nunique())
print("Upstream task recordings:", len(manifest))
print("Source CSV on disk (MiB):", round(source_csv.stat().st_size / 1024**2, 1))
display(manifest[["record_id", "participant_id", "task"]].head(8))


**03. Apply Gap-Respecting Smoothing**

Use a centred **five-sample rolling median** separately inside uninterrupted valid-sample runs. Eligibility requires source `val = 0`, finite `x/y`, increasing timestamp `n`, and no adjacent time gap greater than **5 ms**. Invalid samples, reversed timestamps and longer gaps break continuity; no interpolation or synthetic blink repair is introduced.

The record-wise streaming implementation keeps the original coordinates, timestamps, `sample_index`, source identity, optional pupil values and vendor annotations. New `x_smooth`, `y_smooth`, `segment_id` and `velocity_dva_s` columns are saved alongside them. The five-sample median and 5-ms gap are **initial method settings**, not values selected to maximise agreement with vendor labels.


In [ ]:
# ==================== 03. Apply Gap-Respecting Smoothing =====================

# -------------------- 1. Declare Scientific Settings -------------------------
SMOOTHING_SAMPLES = 5
MAX_GAP_MS = 5.0
READ_CHUNK_ROWS = 80_000

# ------------------ 2. Stream Full Records and Save Output -------------------
segments, example_trace = preprocess_stage_file(
    source_csv,
    output_dir / "02_preprocessed_samples.csv",
    smoothing_samples=SMOOTHING_SAMPLES,
    max_gap_ms=MAX_GAP_MS,
    chunksize=READ_CHUNK_ROWS,
    expected_records=expected_records,
)
if len(segments) != len(manifest):
    raise ValueError("Preprocessing record count differs from the source manifest.")
if int(segments["n_samples"].sum()) <= 0:
    raise ValueError("No source samples were processed.")
print("Prepared records:", len(segments))
print("Prepared samples:", int(segments["n_samples"].sum()))


**04. Inspect Segments and Velocity**

Angular velocity uses the Euclidean displacement between adjacent **smoothed** gaze coordinates divided by their actual positive time difference, in seconds. Results are expressed in degrees of visual angle per second (**dva/s**). Velocity remains undefined at the first eligible sample of each valid run and across interruptions.

The centred median uses neighbouring samples on both sides, so this is an **offline representation**, not a causal real-time estimator. The record summary reports source-row count, valid-row count, number of valid runs and observed velocity statistics. Velocity alone does not establish an independently verified fixation or saccade.


In [ ]:
# ===================== 04. Inspect Segments and Velocity =====================

# ----------------------- 1. Persist Record Summary ---------------------------
segments.to_csv(output_dir / "02_preprocessing_summary.csv", index=False)
display(segments)

# -------------------- 2. Check Observable Coverage ---------------------------
print("Participants represented:", segments["participant_id"].nunique())
print("Source records represented:", segments["record_id"].nunique())
print("Prepared output (MiB):", round(
    (output_dir / "02_preprocessed_samples.csv").stat().st_size / 1024**2, 1
))


**05. Compare Raw and Processed Gaze**

Once executed, inspect up to the first 3,000 samples of a selected recording. The upper plot compares original horizontal gaze against its within-run centred-median representation; the lower plot displays derived velocity, including missing values at boundaries.

The introductory SVG shows the **actual raw input excerpt** and separate **schematic** explanations of segmentation, smoothing and velocity. The plot saved by this cell will provide the actual stage-02 visual evidence; no improvement in event boundaries is presumed before subsequent detector and sensitivity analyses.


In [ ]:
# ==================== 05. Compare Raw and Processed Gaze =====================

record_id = str(example_trace.record_id.iloc[0])
part = example_trace
fig, axes = plt.subplots(2,1, figsize=(11,5), sharex=True)
axes[0].plot(part.time_s, part.x, lw=0.6, label="raw x")
axes[0].plot(part.time_s, part.x_smooth, lw=0.8, label="median-smoothed x")
axes[0].set_ylabel("Horizontal gaze (dva)")
axes[0].legend()
axes[1].plot(part.time_s, part.velocity_dva_s, lw=0.7)
axes[1].set_ylabel("Velocity (dva/s)")
axes[1].set_xlabel("Time from record start (s)")
fig.suptitle(f"Gap-aware motion representation: {record_id}")
fig.tight_layout()
save_figure(fig, output_dir / "02_velocity_profile.png")
plt.show()


**06. Export Reusable Numerical Inputs**

Save the complete, current `02_preprocessed_samples.csv`, the per-record `02_preprocessing_summary.csv`, and the `02_velocity_profile.png` figure. Notebook 03 reads the sample table directly and does not rerun earlier source inspection or preprocessing.

The output CSV and downloaded ZIP may be large; retain them privately for new Colab sessions instead of committing them to GitHub. Inspect real sample/record coverage, valid-run summaries and the plotted velocity before interpreting downstream event labels.


In [ ]:
# =================== 06. Export Reusable Numerical Inputs ====================

print("Missing velocities at segment starts remain NaN.")
print("Current motion representation:", output_dir / "02_preprocessed_samples.csv")
download_outputs_from_colab("eye_tracking_foundations/output", "02")
